# 6. Evaluate (scorecard)

The measurement instrument for tuning (see `../docs/TUNING.md`).
Prints coverage + accuracy per schema/source, and the reason breakdown that
tells you which lever to pull. Accuracy needs `ground_truth`.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openpyxl pydantic
%restart_python

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.evaluate import scorecard, reason_breakdown

spark = get_spark()
config = load_config()

dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
spark.sql(f'USE {catalog}.{schema}')

## Scorecard

Coverage = classified / total. Accuracy = correct / labelled (labelled =
invoices that have ground truth). Move these numbers; that's all tuning is.

In [ ]:
scorecard(spark, catalog, schema).display()

## Reason breakdown

The dominant `reason` among unclassified rows maps to one lever in
`docs/TUNING.md` (`no_candidates` → seed GT / embed taxonomy / raise top_k;
`out_of_taxonomy` → add GT examples / raise top_k / tighten prompt).

In [ ]:
reason_breakdown(spark, catalog, schema).display()